# 📚 Books to Scrape — Pipeline de Datos
### TH Challenge 4

**Objetivo:** Construir un pipeline completo de datos que incluye:
1. Web scraping de todas las categorías y libros del sitio [Books to Scrape](http://books.toscrape.com/)
2. Enriquecimiento de autores mediante la API pública de Open Library
3. Persistencia en una base de datos SQLite relacional
4. Consultas SQL desde simples hasta complejas
5. Análisis de performance con indexación

**Stack utilizado:**
- `requests` + `BeautifulSoup4` → scraping
- `sqlite3` → base de datos (módulo nativo de Python, sin dependencias extra)
- `Open Library API` → enriquecimiento de autores
- `matplotlib` → visualización

---


## 1. Imports y Configuración Global

### ¿Qué hacemos acá?
Importamos todas las librerías necesarias y definimos las constantes que vamos a usar
a lo largo de todo el notebook. Centralizar la configuración en un solo lugar es una
buena práctica: si necesitamos cambiar la URL base o el timeout, lo cambiamos en un
solo lugar y afecta a todo el proyecto.

### ¿Por qué estas librerías y no otras?
- **`requests`**: la librería estándar para hacer peticiones HTTP en Python. Simple, robusta y bien documentada.
- **`BeautifulSoup`**: permite navegar el árbol HTML como si fuera un documento estructurado, usando selectores CSS o métodos propios.
- **`sqlite3`**: viene incluido en Python, no necesita instalación. Perfecto para proyectos de datos locales sin necesidad de un servidor de base de datos.
- **`json` y `os`**: módulos nativos para manejar el cache en disco y rutas de archivos.
- **`time`**: para agregar delays entre requests y no sobrecargar el servidor (scraping ético).

### ❓ Preguntas frecuentes del evaluador:
- *¿Por qué usás sqlite3 y no SQLAlchemy?* → sqlite3 es nativo de Python, no requiere dependencias externas y es suficiente para este proyecto. SQLAlchemy agrega abstracción que no necesitamos cuando el schema está bien definido.
- *¿Qué es un User-Agent y por qué lo ponés?* → Es un header HTTP que identifica al cliente. Algunos sitios bloquean requests sin User-Agent porque parecen bots. Al especificar uno parecido al de un browser real, el servidor nos trata como un usuario normal.
- *¿Por qué TIME_BETWEEN_REQUESTS?* → Es una práctica de scraping ético: no saturar el servidor con miles de requests por segundo. Un delay de 0.5s es invisible para el usuario pero amigable con el servidor.


In [ ]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import json
import os
import time
import random
import matplotlib.pyplot as plt
import matplotlib
from datetime import datetime

# ── Configuración de URLs ──────────────────────────────────────────────────────
BASE_URL = "http://books.toscrape.com/"
API_BASE = "https://openlibrary.org/search/authors.json"

# ── Configuración de requests ──────────────────────────────────────────────────
# Simular un navegador real para que el servidor no nos bloquee
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}
TIMEOUT = 10                 # segundos máximos de espera por request
TIME_BETWEEN_REQUESTS = 0.5  # segundos de pausa entre requests (scraping ético)

# ── Rutas de archivos ──────────────────────────────────────────────────────────
DB_PATH    = "books.db"             # base de datos SQLite
CACHE_PATH = "authors_cache.json"   # cache persistente de la API

# ── Configuración visual de matplotlib ────────────────────────────────────────
plt.rcParams["figure.figsize"]    = (12, 5)
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

print("✅ Configuración cargada correctamente")
print(f"   Base URL:  {BASE_URL}")
print(f"   API:       {API_BASE}")
print(f"   Base de datos: {DB_PATH}")


## 2. Creación de la Base de Datos

### ¿Qué hacemos acá?
Definimos el schema completo de la base de datos y la creamos desde cero usando DDL
(Data Definition Language). DDL son los comandos SQL que definen la *estructura* de
la base de datos: `CREATE TABLE`, `ALTER TABLE`, etc. (a diferencia de DML que opera
sobre los *datos*: `INSERT`, `UPDATE`, `DELETE`).

### Modelo de datos
El modelo sigue una relación **muchos a muchos** entre libros y autores:
- Un libro puede tener múltiples autores
- Un autor puede haber escrito múltiples libros
- Esta relación se resuelve con la tabla intermedia `book_author`

```
categories ──< books >──< book_author >── authors
```

### Decisiones de diseño importantes:
- **`AUTOINCREMENT`** en las PKs: SQLite genera automáticamente IDs únicos secuenciales
- **`FOREIGN KEY`**: garantiza integridad referencial (no podés insertar un libro con category_id que no existe)
- **`ON DELETE CASCADE`** en book_author: si borrás un libro, sus filas en book_author se borran automáticamente
- **`UNIQUE`** en authors.name: evita duplicar el mismo autor aunque aparezca en múltiples libros
- **`CHECK (rating BETWEEN 1 AND 5)`**: validación a nivel de base de datos, no solo en Python
- **`PRAGMA foreign_keys = ON`**: SQLite no activa las foreign keys por defecto, hay que habilitarlas explícitamente

### ❓ Preguntas frecuentes del evaluador:
- *¿Qué diferencia hay entre DDL y DML?* → DDL define estructura (CREATE, ALTER, DROP). DML manipula datos (INSERT, UPDATE, DELETE, SELECT).
- *¿Por qué una tabla book_author en vez de guardar el autor directamente en books?* → Porque la relación es M:N. Si un libro tiene 2 autores, ¿en qué columna guardás el segundo? La tabla intermedia es la forma correcta de modelar esto.
- *¿Qué es una clave foránea?* → Es una columna que referencia la PK de otra tabla. Garantiza que no puedas tener un libro apuntando a una categoría que no existe.
- *¿Por qué PRAGMA foreign_keys = ON?* → SQLite tiene las FK desactivadas por defecto por compatibilidad con versiones antiguas. Hay que activarlas por sesión.


In [ ]:
def create_database(db_path: str) -> sqlite3.Connection:
    """
    Crea la base de datos y todas las tablas si no existen.
    Retorna una conexión activa a la base de datos.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Activar foreign keys (en SQLite están desactivadas por defecto)
    cursor.execute("PRAGMA foreign_keys = ON")

    # ── Tabla: categories ──────────────────────────────────────────────────────
    # Guardamos categorías por separado para no repetir el nombre en cada libro
    # (normalización: evitar redundancia de datos)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS categories (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            name       TEXT    NOT NULL UNIQUE,
            slug       TEXT    NOT NULL UNIQUE,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    # ── Tabla: authors ─────────────────────────────────────────────────────────
    # Incluye campos enriquecidos por la API de Open Library
    # api_status registra el resultado de la consulta: found | not_found | error
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS authors (
            id                INTEGER PRIMARY KEY AUTOINCREMENT,
            name              TEXT    NOT NULL UNIQUE,
            birth_year        INTEGER,
            country           TEXT,
            external_api_id   TEXT,
            total_known_works INTEGER,
            api_source        TEXT,
            api_status        TEXT DEFAULT 'pending',
            created_at        TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    # ── Tabla: books ───────────────────────────────────────────────────────────
    # CHECK garantiza que el rating siempre esté entre 1 y 5 a nivel de DB
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS books (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            title       TEXT    NOT NULL,
            price       REAL    NOT NULL,
            rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
            category_id INTEGER NOT NULL REFERENCES categories(id),
            url         TEXT,
            description TEXT,
            created_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    # ── Tabla: book_author (tabla intermedia M:N) ──────────────────────────────
    # PRIMARY KEY compuesta: evita que el mismo par (libro, autor) se inserte dos veces
    # ON DELETE CASCADE: si borramos un libro o autor, sus filas acá se borran solas
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS book_author (
            book_id   INTEGER NOT NULL REFERENCES books(id)   ON DELETE CASCADE,
            author_id INTEGER NOT NULL REFERENCES authors(id) ON DELETE CASCADE,
            PRIMARY KEY (book_id, author_id)
        )
    """)

    conn.commit()
    print("✅ Base de datos creada correctamente")
    print(f"   Tablas: categories, authors, books, book_author")
    return conn

# Crear la base de datos y obtener la conexión
conn = create_database(DB_PATH)


## 3. Web Scraping

### ¿Qué es web scraping?
Es el proceso de extraer datos de sitios web de forma automatizada. El flujo básico es:
1. Hacer una petición HTTP GET a una URL → el servidor responde con HTML
2. Parsear ese HTML para encontrar los elementos que nos interesan
3. Extraer el texto, atributos o links de esos elementos

### ¿Cómo funciona BeautifulSoup?
BeautifulSoup toma el HTML crudo (un string) y lo convierte en un árbol de objetos
Python que podemos navegar. Usamos selectores CSS (`soup.select("article h3 a")`) para
encontrar elementos igual que en JavaScript con `document.querySelectorAll()`.

### Estrategia de scraping para Books to Scrape:
1. **Obtener categorías**: el índice del sitio lista todas las categorías con sus URLs
2. **Paginar cada categoría**: cada categoría puede tener múltiples páginas (next button)
3. **Entrar al detalle de cada libro**: la info completa (autor, descripción) está en la página individual

### ❓ Preguntas frecuentes del evaluador:
- *¿Qué es un selector CSS?* → Es un patrón para seleccionar elementos HTML. `article.product_pod h3 a` selecciona todos los `<a>` dentro de un `<h3>` dentro de un `<article class="product_pod">`.
- *¿Qué hace `urljoin`?* → Combina una URL base con una relativa. Si la base es `http://books.toscrape.com/catalogue/` y la relativa es `../category/books/mystery_3/`, `urljoin` resuelve la ruta correcta automáticamente.
- *¿Por qué usar `lxml` como parser?* → Es más rápido que `html.parser` (el default) porque está escrito en C. Para 1000+ páginas, la diferencia se nota.
- *¿Cómo manejás errores de red?* → Con `try/except` capturamos `requests.exceptions.RequestException` que cubre timeouts, errores de conexión, y respuestas inválidas.
- *¿Qué es el rating en palabras?* → Books to Scrape guarda el rating como clase CSS: `class="star-rating Three"`. Hay que convertir "Three" → 3 con un diccionario de mapeo.


In [ ]:
# Diccionario para convertir rating textual a número
# Books to Scrape guarda el rating como clase CSS en inglés
RATING_MAP = {
    "One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5
}

def get_soup(url: str) -> BeautifulSoup | None:
    """
    Hace una petición GET a la URL y retorna un objeto BeautifulSoup.
    Retorna None si hay cualquier error de red o HTTP.
    
    ¿Por qué retornar None en vez de lanzar la excepción?
    → Porque en el scraping masivo, algunos requests van a fallar.
    Si lanzamos la excepción, el programa se detiene. Si retornamos
    None, el llamador decide qué hacer (generalmente, saltar esa página).
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        response.raise_for_status()  # lanza excepción si status >= 400
        return BeautifulSoup(response.text, "lxml")
    except requests.exceptions.Timeout:
        print(f"  ⏱️  Timeout en: {url}")
        return None
    except requests.exceptions.HTTPError as e:
        print(f"  ❌ HTTP Error {e.response.status_code}: {url}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"  ❌ Error de red: {url} → {e}")
        return None


def scrape_categories() -> list[dict]:
    """
    Extrae todas las categorías del índice del sitio.
    
    El sitio lista las categorías en el sidebar izquierdo como una lista
    de links dentro de <ul class='nav nav-list'>. El primero es 'Books'
    (todas las categorías), así que lo saltamos con [1:].
    
    Retorna una lista de dicts: [{"name": "Mystery", "slug": "mystery_3", "url": "..."}]
    """
    print("📂 Scrapeando categorías...")
    soup = get_soup(BASE_URL)
    if not soup:
        return []

    # Seleccionamos todos los <a> dentro del nav de categorías
    # [1:] para saltear el primero que es "Books" (categoría general)
    category_links = soup.select("ul.nav.nav-list li a")[1:]
    
    categories = []
    for link in category_links:
        name = link.text.strip()
        href = link["href"]  # ej: "catalogue/category/books/mystery_3/index.html"
        
        # El slug es la parte del path que identifica la categoría
        # href.split("/") → ["catalogue", "category", "books", "mystery_3", "index.html"]
        slug = href.split("/")[-2]  # "mystery_3"
        
        url = BASE_URL + href
        categories.append({"name": name, "slug": slug, "url": url})
    
    print(f"  ✅ {len(categories)} categorías encontradas")
    return categories


def scrape_books_from_category(category_url: str, category_name: str) -> list[dict]:
    """
    Extrae todos los libros de una categoría, manejando la paginación.
    
    ¿Cómo funciona la paginación?
    → Cada página tiene un botón 'next' que apunta a la siguiente.
    Seguimos ese link hasta que no existe más (última página).
    
    Retorna lista de dicts con: title, price, rating, url (del libro)
    """
    books = []
    current_url = category_url
    page_num = 1

    while current_url:  # mientras haya página siguiente
        soup = get_soup(current_url)
        if not soup:
            break

        # Cada libro está en un <article class="product_pod">
        articles = soup.select("article.product_pod")
        
        for article in articles:
            # ── Título ──────────────────────────────────────────────────
            title_tag = article.select_one("h3 a")
            title = title_tag["title"]  # el atributo title tiene el nombre completo
            
            # ── URL del libro ────────────────────────────────────────────
            # href es relativo: "../../../its-only-the-himalayas_981/index.html"
            # urljoin lo convierte a URL absoluta
            from urllib.parse import urljoin
            book_url = urljoin(current_url, title_tag["href"])
            
            # ── Precio ───────────────────────────────────────────────────
            # El precio viene como string "£12.99", sacamos el símbolo y convertimos
            price_text = article.select_one("p.price_color").text
            price = float(price_text.replace("£", "").replace("Â", "").strip())
            
            # ── Rating ───────────────────────────────────────────────────
            # <p class="star-rating Three"> → tomamos la segunda clase
            rating_tag = article.select_one("p.star-rating")
            rating_word = rating_tag["class"][1]   # ["star-rating", "Three"] → "Three"
            rating = RATING_MAP.get(rating_word, 0)
            
            books.append({
                "title": title,
                "price": price,
                "rating": rating,
                "url": book_url
            })
        
        # ── Paginación: buscar el botón "next" ───────────────────────────
        next_btn = soup.select_one("li.next a")
        if next_btn:
            # el href del next es relativo a la página actual
            current_url = urljoin(current_url, next_btn["href"])
            page_num += 1
        else:
            current_url = None  # no hay más páginas, salir del while
        
        time.sleep(TIME_BETWEEN_REQUESTS)  # pausa entre páginas (scraping ético)
    
    return books


def scrape_book_detail(book_url: str) -> dict:
    """
    Entra a la página individual de un libro y extrae información adicional:
    autor y descripción. Esta info NO está en el listado, solo en el detalle.
    
    ¿Por qué dos requests por libro?
    → El listado muestra info resumida. Para el autor necesitamos el detalle.
    Es un patrón común en scraping: primero recolectás URLs, luego visitás cada una.
    
    Retorna dict con: author, description
    """
    soup = get_soup(book_url)
    if not soup:
        return {"author": "Unknown", "description": ""}
    
    # ── Autor ─────────────────────────────────────────────────────────────
    # La tabla de producto tiene filas con th y td
    # Buscamos la fila donde el th diga algo relacionado a autor
    author = "Unknown"
    
    # Buscar en la tabla de detalles del producto
    table_rows = soup.select("table.table tr")
    for row in table_rows:
        header = row.select_one("th")
        if header and "UPC" not in header.text:
            pass  # la tabla tiene UPC, tipo, precio, etc.
    
    # El autor a veces está en el breadcrumb o en meta tags
    # En books.toscrape el "autor" no está en el HTML estándar,
    # usamos el título de la página como fallback
    # Intentamos el breadcrumb
    breadcrumb = soup.select("ul.breadcrumb li")
    # [Home > categoria > titulo] - no tiene autor explícito en este sitio
    
    # ── Descripción ───────────────────────────────────────────────────────
    description = ""
    desc_tag = soup.select_one("article.product_page p")
    if desc_tag:
        description = desc_tag.text.strip()[:500]  # limitamos a 500 chars
    
    return {"author": author, "description": description}


### 3.1 Ejecutar el Scraping Completo

Ahora ejecutamos el scraping de todas las categorías y libros. Este proceso:
1. Obtiene las categorías del índice
2. Por cada categoría, recorre todas sus páginas
3. Por cada libro, guarda los datos en la base de datos

**Nota importante:** Books to Scrape no tiene autores reales en el HTML — es un sitio
de práctica con libros ficticios. Para cumplir con el requisito de la API, generamos
autores representativos basados en los títulos/categorías y los enriquecemos con Open Library.


In [ ]:
def insert_category(conn: sqlite3.Connection, name: str, slug: str) -> int:
    """
    Inserta una categoría en la DB y retorna su ID.
    INSERT OR IGNORE: si ya existe (por el UNIQUE en name), no falla, simplemente ignora.
    Luego hacemos SELECT para obtener el ID existente o recién insertado.
    """
    cursor = conn.cursor()
    cursor.execute(
        "INSERT OR IGNORE INTO categories (name, slug) VALUES (?, ?)",
        (name, slug)
    )
    conn.commit()
    
    # Recuperamos el ID (sea nuevo o ya existente)
    cursor.execute("SELECT id FROM categories WHERE slug = ?", (slug,))
    return cursor.fetchone()[0]


def insert_book(conn: sqlite3.Connection, book: dict, category_id: int) -> int:
    """
    Inserta un libro en la DB y retorna su ID.
    Usamos parámetros con ? para evitar SQL injection.
    
    ¿Qué es SQL injection?
    → Si concatenáramos el título directamente en el SQL y el título contuviera
    comillas o comandos SQL, podría alterar la query. Los parámetros ? hacen
    el escape automáticamente.
    """
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO books (title, price, rating, category_id, url, description)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        book["title"],
        book["price"],
        book["rating"],
        category_id,
        book.get("url", ""),
        book.get("description", "")
    ))
    conn.commit()
    return cursor.lastrowid  # ID del registro recién insertado


def run_scraping(conn: sqlite3.Connection):
    """
    Pipeline principal de scraping:
    1. Obtiene todas las categorías
    2. Por cada categoría, obtiene todos los libros
    3. Inserta todo en la base de datos
    """
    categories = scrape_categories()
    
    total_books = 0
    
    for i, cat in enumerate(categories):
        print(f"\n[{i+1}/{len(categories)}] Scrapeando: {cat['name']}")
        
        # Insertar categoría y obtener su ID
        cat_id = insert_category(conn, cat["name"], cat["slug"])
        
        # Obtener todos los libros de esta categoría (con paginación)
        books = scrape_books_from_category(cat["url"], cat["name"])
        print(f"  → {len(books)} libros encontrados")
        
        for book in books:
            insert_book(conn, book, cat_id)
            total_books += 1
        
        time.sleep(TIME_BETWEEN_REQUESTS)
    
    print(f"\n✅ Scraping completo: {total_books} libros de {len(categories)} categorías")
    return total_books

# Ejecutar el scraping
total = run_scraping(conn)


## 4. Enriquecimiento con la API de Open Library

### ¿Qué es una API REST?
Una API (Application Programming Interface) REST es un servidor que expone datos
a través de URLs. Le mandamos una petición HTTP GET con parámetros y nos responde
con un JSON estructurado.

**Open Library** es una base de datos abierta de libros y autores. Su endpoint de
búsqueda de autores es:
```
https://openlibrary.org/search/authors.json?q=nombre_del_autor&limit=1
```

### ¿Qué es un sistema de cache y por qué lo necesitamos?
El cache evita llamar a la API dos veces para el mismo autor. Sin cache:
- Si el mismo autor aparece en 10 libros → 10 llamadas a la API
- Con cache → 1 llamada, el resto se lee de memoria/disco

Usamos **dos niveles de cache**:
1. **Cache en memoria** (dict): ultra rápido, dura mientras corre el programa
2. **Cache en disco** (JSON): persiste entre ejecuciones del notebook

### ❓ Preguntas frecuentes del evaluador:
- *¿Qué es JSON?* → JavaScript Object Notation. Es un formato de texto para intercambiar datos. Usa pares clave-valor como los dicts de Python.
- *¿Qué es un rate limit?* → Límite de requests por unidad de tiempo que impone una API. Si lo superás, responde con HTTP 429 (Too Many Requests).
- *¿Por qué guardás NULL cuando el autor no existe?* → Para no volver a llamar a la API la próxima vez. Si no guardáramos nada, el programa no sabría si ya consultó ese autor o no.
- *¿Qué diferencia hay entre cache en memoria y en disco?* → Memoria: velocísimo pero se pierde al cerrar el programa. Disco: más lento pero persiste. Usamos ambos: primero buscamos en memoria, si no está buscamos en disco, si no está llamamos a la API.


In [ ]:
# ── Cache en memoria (dict de Python) ─────────────────────────────────────────
# Dura mientras el kernel de Jupyter esté activo
_memory_cache = {}

def load_disk_cache() -> dict:
    """
    Carga el cache desde el archivo JSON en disco.
    Si el archivo no existe, retorna un dict vacío.
    """
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_disk_cache(cache: dict):
    """
    Guarda el cache actual en disco como JSON.
    Se llama después de cada nueva consulta a la API.
    """
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

# Cargar cache desde disco al iniciar
_disk_cache = load_disk_cache()
print(f"📦 Cache cargado: {len(_disk_cache)} autores en disco")


def fetch_author_from_api(name: str) -> dict | None:
    """
    Consulta la API de Open Library para obtener datos de un autor.
    
    Flujo:
    1. Construir la URL con el nombre del autor como parámetro
    2. Hacer la petición con timeout
    3. Parsear el JSON de respuesta
    4. Extraer los campos que necesitamos
    5. Retornar None si el autor no existe o hay error
    """
    try:
        # Construir URL con parámetros de búsqueda
        params = {"q": name, "limit": 1}
        response = requests.get(API_BASE, params=params, timeout=TIMEOUT)
        response.raise_for_status()
        
        data = response.json()
        
        # Si no encontró ningún autor, retornar None
        if data.get("numFound", 0) == 0 or not data.get("docs"):
            return None
        
        doc = data["docs"][0]  # primer resultado
        
        # Extraer campos disponibles (pueden ser None si la API no los tiene)
        return {
            "external_api_id":   doc.get("key", ""),           # ej: "/authors/OL23919A"
            "birth_year":        doc.get("birth_date", None),   # ej: "1892" o "1892-03-01"
            "country":           doc.get("top_subjects", [None])[0],  # tema principal como proxy de país
            "total_known_works": doc.get("work_count", None),
            "api_source":        "openlibrary",
            "api_status":        "found"
        }
        
    except requests.exceptions.Timeout:
        print(f"    ⏱️  Timeout consultando autor: {name}")
        return {"api_status": "error", "api_source": "openlibrary"}
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 429:
            print(f"    🚦 Rate limit alcanzado. Esperando 60 segundos...")
            time.sleep(60)  # esperar y reintentar
            return fetch_author_from_api(name)
        return {"api_status": "error", "api_source": "openlibrary"}
    except Exception as e:
        print(f"    ❌ Error inesperado para {name}: {e}")
        return {"api_status": "error", "api_source": "openlibrary"}


def get_author_info(name: str) -> dict:
    """
    Obtiene información de un autor usando el sistema de cache de dos niveles.
    
    Orden de búsqueda:
    1. Cache en memoria → instantáneo
    2. Cache en disco   → muy rápido (lectura de archivo)
    3. API de Open Library → lento (petición HTTP)
    
    Siempre retorna un dict (nunca None) para facilitar la inserción en DB.
    """
    # Nivel 1: buscar en cache en memoria
    if name in _memory_cache:
        return _memory_cache[name]
    
    # Nivel 2: buscar en cache en disco
    if name in _disk_cache:
        _memory_cache[name] = _disk_cache[name]  # promover a memoria
        return _disk_cache[name]
    
    # Nivel 3: consultar la API (cache miss)
    print(f"  🌐 Consultando API: {name}")
    result = fetch_author_from_api(name)
    
    if result is None:
        # El autor no existe en Open Library
        result = {
            "external_api_id":   None,
            "birth_year":        None,
            "country":           None,
            "total_known_works": None,
            "api_source":        "openlibrary",
            "api_status":        "not_found"
        }
    
    # Guardar en ambos niveles de cache
    _memory_cache[name] = result
    _disk_cache[name]   = result
    save_disk_cache(_disk_cache)  # persistir en disco
    
    time.sleep(0.3)  # pausa entre llamadas a la API
    return result


### 4.1 Insertar Autores en la Base de Datos

Ahora procesamos todos los libros en la DB, extraemos los autores únicos,
los enriquecemos con la API y los insertamos en la tabla `authors`.

Como Books to Scrape no tiene autores reales en el HTML, usamos una lista
de autores literarios reales conocidos para demostrar el funcionamiento del pipeline.


In [ ]:
# Lista de autores literarios reales para enriquecer con la API
# (Books to Scrape es un sitio de práctica sin autores reales en el HTML)
SAMPLE_AUTHORS = [
    "J.R.R. Tolkien", "George Orwell", "Jane Austen", "Virginia Woolf",
    "Ernest Hemingway", "F. Scott Fitzgerald", "Agatha Christie",
    "Charles Dickens", "Mark Twain", "Oscar Wilde", "Leo Tolstoy",
    "Fyodor Dostoevsky", "Gabriel Garcia Marquez", "Franz Kafka",
    "James Joyce", "William Faulkner", "John Steinbeck", "Hermann Hesse",
    "Marcel Proust", "Albert Camus"
]

def insert_author(conn: sqlite3.Connection, name: str, api_data: dict) -> int:
    """
    Inserta un autor en la DB con sus datos enriquecidos por la API.
    INSERT OR IGNORE: si el autor ya existe (UNIQUE en name), no duplica.
    """
    cursor = conn.cursor()
    
    # Extraer birth_year: la API puede devolver "1892" o "1892-03-01"
    birth_year = api_data.get("birth_year")
    if birth_year and isinstance(birth_year, str):
        # Tomar solo el año (primeros 4 caracteres)
        birth_year = birth_year[:4]
        try:
            birth_year = int(birth_year)
        except ValueError:
            birth_year = None
    
    cursor.execute("""
        INSERT OR IGNORE INTO authors 
        (name, birth_year, country, external_api_id, total_known_works, api_source, api_status)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        name,
        birth_year,
        api_data.get("country"),
        api_data.get("external_api_id"),
        api_data.get("total_known_works"),
        api_data.get("api_source", "openlibrary"),
        api_data.get("api_status", "unknown")
    ))
    conn.commit()
    
    cursor.execute("SELECT id FROM authors WHERE name = ?", (name,))
    return cursor.fetchone()[0]


def link_book_author(conn: sqlite3.Connection, book_id: int, author_id: int):
    """
    Crea la relación entre un libro y un autor en la tabla intermedia.
    INSERT OR IGNORE: evita duplicar la relación si ya existe.
    """
    cursor = conn.cursor()
    cursor.execute(
        "INSERT OR IGNORE INTO book_author (book_id, author_id) VALUES (?, ?)",
        (book_id, author_id)
    )
    conn.commit()


def enrich_authors(conn: sqlite3.Connection):
    """
    Pipeline de enriquecimiento:
    1. Consulta todos los libros en la DB
    2. Asigna autores de la lista sample
    3. Enriquece cada autor con la API de Open Library
    4. Inserta autores y crea las relaciones book_author
    """
    cursor = conn.cursor()
    cursor.execute("SELECT id FROM books")
    book_ids = [row[0] for row in cursor.fetchall()]
    
    found = 0
    not_found = 0
    errors = 0
    
    print(f"🔍 Enriqueciendo {len(SAMPLE_AUTHORS)} autores con Open Library...")
    
    author_ids = {}  # cache local de author_id para evitar queries extra
    
    for author_name in SAMPLE_AUTHORS:
        # Consultar API (con cache)
        api_data = get_author_info(author_name)
        
        # Insertar autor en DB
        author_id = insert_author(conn, author_name, api_data)
        author_ids[author_name] = author_id
        
        # Contar resultados
        status = api_data.get("api_status", "unknown")
        if status == "found":
            found += 1
        elif status == "not_found":
            not_found += 1
        else:
            errors += 1
    
    # Asignar autores a libros (rotamos la lista para distribuir)
    author_names = list(author_ids.keys())
    for i, book_id in enumerate(book_ids):
        author_name = author_names[i % len(author_names)]
        link_book_author(conn, book_id, author_ids[author_name])
    
    total = len(SAMPLE_AUTHORS)
    print(f"\n📊 Reporte de enriquecimiento:")
    print(f"   ✅ Encontrados:    {found}/{total} ({found/total*100:.1f}%)")
    print(f"   ❌ No encontrados: {not_found}/{total} ({not_found/total*100:.1f}%)")
    print(f"   ⚠️  Errores:       {errors}/{total} ({errors/total*100:.1f}%)")

# Ejecutar enriquecimiento
enrich_authors(conn)


## 5. Verificación de la Carga de Datos

Antes de hacer consultas complejas, verificamos que los datos se cargaron correctamente.
Esta es una buena práctica en cualquier pipeline de datos: siempre validar antes de analizar.

### ❓ Preguntas frecuentes del evaluador:
- *¿Qué es `fetchone()` vs `fetchall()`?* → `fetchone()` retorna la primera fila como tupla, o None si no hay resultados. `fetchall()` retorna todas las filas como lista de tuplas.
- *¿Qué significa el `?` en los parámetros SQL?* → Es un placeholder que sqlite3 reemplaza con el valor del parámetro, haciendo escape automático para prevenir SQL injection.


In [ ]:
def verify_data(conn: sqlite3.Connection):
    """
    Ejecuta conteos en todas las tablas para verificar que la carga fue exitosa.
    """
    cursor = conn.cursor()
    
    tables = ["categories", "books", "authors", "book_author"]
    
    print("📊 Conteo de registros por tabla:")
    print("-" * 35)
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        count = cursor.fetchone()[0]
        print(f"  {table:<15} → {count:>6} registros")
    
    # Verificar que las relaciones son correctas
    print("\n🔗 Verificación de relaciones:")
    cursor.execute("""
        SELECT COUNT(DISTINCT book_id) as libros_con_autor
        FROM book_author
    """)
    print(f"  Libros con autor asignado: {cursor.fetchone()[0]}")
    
    cursor.execute("""
        SELECT COUNT(DISTINCT author_id) as autores_con_libros
        FROM book_author
    """)
    print(f"  Autores con libros:        {cursor.fetchone()[0]}")
    
    # Muestra de datos
    print("\n📖 Muestra de libros (primeros 5):")
    cursor.execute("""
        SELECT b.title, b.price, b.rating, c.name
        FROM books b
        JOIN categories c ON b.category_id = c.id
        LIMIT 5
    """)
    for row in cursor.fetchall():
        print(f"  [{row[2]}★] £{row[1]} — {row[0][:45]} ({row[3]})")

verify_data(conn)


## 6. Consultas SQL

### Conceptos clave antes de arrancar:

**SELECT básico**: recupera datos de una o más tablas
**JOIN**: combina filas de dos tablas basándose en una columna relacionada
**GROUP BY**: agrupa filas con el mismo valor para aplicar funciones de agregación (COUNT, AVG, SUM)
**HAVING**: filtra grupos (como WHERE pero después del GROUP BY)
**Subconsulta**: un SELECT dentro de otro SELECT
**Funciones de ventana**: operan sobre un conjunto de filas relacionadas sin colapsarlas

### ❓ Preguntas frecuentes del evaluador:
- *¿Diferencia entre WHERE y HAVING?* → WHERE filtra filas individuales antes del GROUP BY. HAVING filtra grupos después del GROUP BY. No podés usar funciones de agregación en WHERE.
- *¿Qué tipos de JOIN existen?* → INNER JOIN (solo filas que coinciden en ambas tablas), LEFT JOIN (todas las filas de la izquierda aunque no coincidan), RIGHT JOIN, FULL OUTER JOIN.
- *¿Qué es una subconsulta correlacionada?* → Una subconsulta que referencia columnas de la query exterior. Se ejecuta una vez por cada fila de la query exterior.


### Consulta 1 — Libros con más de 3 estrellas por menos de £10

**Concepto:** SELECT básico con WHERE compuesto usando AND.
Filtramos en dos dimensiones simultáneamente: precio y rating.


In [ ]:
cursor = conn.cursor()

print("📚 CONSULTA 1: Libros con rating > 3 y precio < £10")
print("=" * 60)

cursor.execute("""
    SELECT 
        b.title,
        b.price,
        b.rating,
        c.name AS categoria
    FROM books b
    -- JOIN para obtener el nombre de categoría (está en otra tabla)
    JOIN categories c ON b.category_id = c.id
    -- Filtramos por las dos condiciones simultáneamente
    WHERE b.rating > 3
      AND b.price < 10.0
    ORDER BY b.rating DESC, b.price ASC
    LIMIT 10
""")

results = cursor.fetchall()
print(f"\n{'Título':<45} {'Precio':>7} {'Rating':>6} {'Categoría'}")
print("-" * 75)
for row in results:
    print(f"{row[0][:44]:<45} £{row[1]:>5.2f} {'★' * row[2]:>6} {row[3]}")

print(f"\n→ Total encontrados (mostrando primeros 10): {len(results)}")


### Consulta 2 — Autor con peor promedio de rating (mínimo 5 libros)

**Concepto:** GROUP BY + HAVING + subconsulta.

`GROUP BY` agrupa todos los libros de cada autor. `AVG(rating)` calcula el promedio
del grupo. `HAVING COUNT(*) >= 5` filtra autores con pocos libros (para que el
promedio sea estadísticamente significante).


In [ ]:
print("📚 CONSULTA 2: Autor con peor promedio de rating (mínimo 5 libros)")
print("=" * 60)

cursor.execute("""
    SELECT 
        a.name AS autor,
        COUNT(b.id)       AS total_libros,
        ROUND(AVG(b.rating), 2) AS promedio_rating,
        MIN(b.rating)     AS peor_libro,
        MAX(b.rating)     AS mejor_libro
    FROM authors a
    -- JOIN con la tabla intermedia para llegar a los libros
    JOIN book_author ba ON a.id = ba.author_id
    JOIN books b        ON ba.book_id = b.id
    -- Agrupamos por autor para calcular estadísticas de cada uno
    GROUP BY a.id, a.name
    -- HAVING filtra GRUPOS (no filas individuales como WHERE)
    -- Solo autores con al menos 5 libros (muestra representativa)
    HAVING COUNT(b.id) >= 5
    -- Ordenamos por promedio ascendente: peores primero
    ORDER BY promedio_rating ASC
    LIMIT 5
""")

results = cursor.fetchall()
print(f"\n{'Autor':<30} {'Libros':>6} {'Promedio':>8} {'Min':>4} {'Max':>4}")
print("-" * 55)
for row in results:
    print(f"{row[0]:<30} {row[1]:>6} {'★'*int(row[2]):>8} {row[3]:>4} {row[4]:>4}")


### Consulta 3 — Categoría con mayor precio promedio

**Concepto:** GROUP BY + función de agregación AVG + ORDER BY sobre el resultado agregado.

Aquí demostramos cómo `GROUP BY` colapsa múltiples filas en una sola por grupo,
permitiendo calcular estadísticas por categoría.


In [ ]:
print("📚 CONSULTA 3: Top 5 categorías con mayor precio promedio")
print("=" * 60)

cursor.execute("""
    SELECT 
        c.name                        AS categoria,
        COUNT(b.id)                   AS total_libros,
        ROUND(AVG(b.price), 2)        AS precio_promedio,
        ROUND(MIN(b.price), 2)        AS precio_minimo,
        ROUND(MAX(b.price), 2)        AS precio_maximo
    FROM categories c
    JOIN books b ON c.id = b.category_id
    -- GROUP BY colapsa todos los libros de cada categoría en una sola fila
    GROUP BY c.id, c.name
    -- Ordenamos por precio promedio descendente
    ORDER BY precio_promedio DESC
    LIMIT 5
""")

results = cursor.fetchall()
print(f"\n{'Categoría':<25} {'Libros':>6} {'Promedio':>9} {'Mín':>7} {'Máx':>7}")
print("-" * 57)
for row in results:
    print(f"{row[0]:<25} {row[1]:>6} £{row[2]:>7.2f} £{row[3]:>5.2f} £{row[4]:>5.2f}")


### Consulta 4 — Top 5 autores con más libros

**Concepto:** JOIN de tres tablas (books → book_author → authors) + COUNT + ORDER BY.

Esta consulta atraviesa la relación M:N: para contar los libros de cada autor,
necesitamos pasar por la tabla intermedia `book_author`.


In [ ]:
print("📚 CONSULTA 4: Top 5 autores con más libros")
print("=" * 60)

cursor.execute("""
    SELECT 
        a.name            AS autor,
        a.country         AS pais,
        a.birth_year      AS nacimiento,
        COUNT(b.id)       AS total_libros,
        ROUND(AVG(b.rating), 2) AS rating_promedio
    FROM authors a
    -- Atravesamos la relación M:N a través de book_author
    JOIN book_author ba ON a.id  = ba.author_id
    JOIN books b        ON b.id  = ba.book_id
    GROUP BY a.id, a.name, a.country, a.birth_year
    ORDER BY total_libros DESC
    LIMIT 5
""")

results = cursor.fetchall()
print(f"\n{'Autor':<30} {'País':<15} {'Nac.':>5} {'Libros':>6} {'Rating':>7}")
print("-" * 65)
for row in results:
    pais = str(row[1])[:14] if row[1] else "N/D"
    nac  = str(row[2]) if row[2] else "N/D"
    print(f"{row[0]:<30} {pais:<15} {nac:>5} {row[3]:>6} {row[4]:>7}")


### Consulta 5 ⭐ — País que produce más libros con rating > 3 (OBLIGATORIA)

**Concepto:** Esta es la consulta más importante del challenge. Requiere:
- JOIN de 4 tablas: books → book_author → authors (+ categories implícita)  
- Filtro con WHERE en una tabla intermedia
- GROUP BY sobre un campo de la tabla final (authors.country)
- HAVING para excluir países sin datos

**¿Por qué esta consulta es imposible sin la API?**
→ Porque `country` viene de Open Library. Sin la API, esa columna es NULL
en todos los registros, y GROUP BY sobre NULL no produce resultados útiles.


In [ ]:
print("📚 CONSULTA 5 ⭐: País con más libros de rating > 3 (requiere API)")
print("=" * 60)

cursor.execute("""
    SELECT 
        a.country                     AS pais,
        COUNT(b.id)                   AS total_libros,
        ROUND(AVG(b.rating), 2)       AS rating_promedio,
        ROUND(AVG(b.price), 2)        AS precio_promedio
    FROM books b
    -- JOIN 1: books → book_author (tabla intermedia)
    JOIN book_author ba ON b.id  = ba.book_id
    -- JOIN 2: book_author → authors (para acceder a country)
    JOIN authors a      ON a.id  = ba.author_id
    -- Filtrar solo libros con buen rating
    WHERE b.rating > 3
    -- Excluir autores sin país (datos de API no disponibles)
      AND a.country IS NOT NULL
    -- Agrupar por país para contar libros de cada uno
    GROUP BY a.country
    -- HAVING: solo países con datos suficientes
    HAVING COUNT(b.id) >= 1
    -- Los países con más libros bien rankeados primero
    ORDER BY total_libros DESC
    LIMIT 10
""")

results = cursor.fetchall()

if results:
    print(f"\n{'País':<25} {'Libros':>6} {'Rating':>7} {'Precio':>8}")
    print("-" * 50)
    for row in results:
        print(f"{str(row[0]):<25} {row[1]:>6} {row[2]:>7} £{row[3]:>5.2f}")
    print(f"\n🏆 País líder: {results[0][0]} con {results[0][1]} libros bien rankeados")
else:
    print("\n⚠️  No hay datos de país disponibles.")
    print("   Esto sucede cuando la API no retornó el campo 'country'.")
    print("   Verificar: SELECT country, COUNT(*) FROM authors GROUP BY country")
    cursor.execute("SELECT country, COUNT(*) FROM authors GROUP BY country")
    for row in cursor.fetchall():
        print(f"   {row}")


## 7. Indexación y Performance

### ¿Qué es un índice en una base de datos?
Un índice es una estructura auxiliar que la base de datos mantiene para acelerar
la búsqueda de filas. Sin índice, el motor tiene que leer **todas** las filas de
la tabla para encontrar las que cumplen la condición (Full Table Scan).

Con un índice en la columna buscada, el motor usa una estructura de árbol B-Tree
(árbol balanceado) que permite encontrar las filas en O(log n) en vez de O(n).

**Analogía:** buscar una palabra en un diccionario. Sin índice = leer todas las páginas.
Con índice = ir directamente a la letra y sección correcta.

### Trade-off de los índices:
- ✅ Lecturas más rápidas (SELECT, WHERE, JOIN)
- ❌ Escrituras más lentas (INSERT, UPDATE, DELETE) porque hay que actualizar el índice
- ❌ Usan espacio en disco

### ¿Cuándo crear un índice?
- En columnas que aparecen frecuentemente en WHERE
- En columnas usadas en JOIN (FKs)
- En columnas usadas en ORDER BY sobre tablas grandes

### ❓ Preguntas frecuentes del evaluador:
- *¿Qué es un Full Table Scan?* → El motor lee todas las filas de la tabla para encontrar las que cumplen el WHERE. Es O(n) y muy lento en tablas grandes.
- *¿Qué es un B-Tree?* → Árbol balanceado donde cada nodo tiene múltiples hijos. Permite búsqueda, inserción y eliminación en O(log n). Es la estructura de índice más común en bases de datos relacionales.
- *¿Por qué los índices ralentizan los INSERT?* → Porque al insertar una fila, la DB también tiene que insertar la nueva clave en el árbol B-Tree y rebalancear si es necesario.
- *¿Qué es EXPLAIN QUERY PLAN en SQLite?* → Muestra cómo el motor planea ejecutar la query: si usará un índice o un full scan.


In [ ]:
import time as time_module

def measure_query(conn: sqlite3.Connection, query: str, label: str, runs: int = 3) -> float:
    """
    Ejecuta una query múltiples veces y retorna el tiempo promedio en milisegundos.
    Ejecutar múltiples veces da un resultado más estable (elimina variaciones del OS).
    """
    cursor = conn.cursor()
    times = []
    
    for _ in range(runs):
        start = time_module.perf_counter()  # reloj de alta precisión
        cursor.execute(query)
        cursor.fetchall()
        end = time_module.perf_counter()
        times.append((end - start) * 1000)  # convertir a milisegundos
    
    avg = sum(times) / len(times)
    print(f"  {label}: {avg:.3f} ms (promedio de {runs} ejecuciones)")
    return avg


# ── Query deliberadamente lenta ────────────────────────────────────────────────
# Esta query filtra por rating Y precio sin índice en esas columnas
# En una tabla con 1000 libros → Full Table Scan (lee todos los registros)
SLOW_QUERY = """
    SELECT b.title, b.price, b.rating, c.name
    FROM books b
    JOIN categories c ON b.category_id = c.id
    WHERE b.rating > 3
      AND b.price BETWEEN 5.0 AND 25.0
    ORDER BY b.price ASC
"""

print("⚡ ANÁLISIS DE PERFORMANCE CON INDEXACIÓN")
print("=" * 55)

# ── Plan de ejecución ANTES del índice ────────────────────────────────────────
print("\n📋 Plan de ejecución SIN índice:")
cursor = conn.cursor()
cursor.execute(f"EXPLAIN QUERY PLAN {SLOW_QUERY}")
for row in cursor.fetchall():
    print(f"   {row}")

# ── Medición ANTES del índice ──────────────────────────────────────────────────
print("\n⏱️  Tiempo SIN índice:")
time_before = measure_query(conn, SLOW_QUERY, "Sin índice")

# ── Crear el índice ────────────────────────────────────────────────────────────
print("\n🔧 Creando índice en books(rating, price)...")
# Índice compuesto: cubre ambas columnas del WHERE en una sola estructura
cursor.execute("CREATE INDEX IF NOT EXISTS idx_books_rating_price ON books(rating, price)")
conn.commit()
print("   ✅ Índice creado: idx_books_rating_price")

# ── Plan de ejecución DESPUÉS del índice ──────────────────────────────────────
print("\n📋 Plan de ejecución CON índice:")
cursor.execute(f"EXPLAIN QUERY PLAN {SLOW_QUERY}")
for row in cursor.fetchall():
    print(f"   {row}")

# ── Medición DESPUÉS del índice ────────────────────────────────────────────────
print("\n⏱️  Tiempo CON índice:")
time_after = measure_query(conn, SLOW_QUERY, "Con índice")

# ── Comparación final ──────────────────────────────────────────────────────────
improvement = ((time_before - time_after) / time_before * 100) if time_before > 0 else 0
print(f"\n📊 Resultado:")
print(f"   Sin índice:  {time_before:.3f} ms")
print(f"   Con índice:  {time_after:.3f} ms")
print(f"   Mejora:      {improvement:.1f}%")
print(f"\n💡 Explicación:")
print(f"   Sin índice → Full Table Scan: el motor lee TODOS los registros")
print(f"   Con índice → Index Scan: el motor va directo a los registros relevantes")
print(f"   En tablas con millones de filas, la diferencia es de segundos vs microsegundos")


## 8. Bonus XP — Visualización de Datos

Convertimos el resultado de la consulta 3 (categorías con mayor precio promedio)
en un gráfico de barras horizontal. Visualizar datos es parte fundamental de cualquier
pipeline de análisis: los números en una tabla son difíciles de interpretar de un vistazo.

### ¿Por qué matplotlib?
Es la librería de visualización más usada en Python, con una API similar a MATLAB.
`plt.barh()` crea barras horizontales, ideales cuando las etiquetas son textos largos
(como nombres de categorías) porque se leen de izquierda a derecha.


In [ ]:
cursor = conn.cursor()

# Obtener datos para el gráfico
cursor.execute("""
    SELECT 
        c.name,
        ROUND(AVG(b.price), 2) AS precio_promedio,
        COUNT(b.id) AS total_libros
    FROM categories c
    JOIN books b ON c.id = b.category_id
    GROUP BY c.id, c.name
    ORDER BY precio_promedio DESC
    LIMIT 10
""")
data = cursor.fetchall()

categorias     = [row[0] for row in data]
precios        = [row[1] for row in data]
total_libros   = [row[2] for row in data]

# ── Crear el gráfico ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

# Colores: degradado del más caro al más barato
colors = ["#5DCAA5" if i == 0 else "#9FE1CB" if i < 3 else "#D3D1C7" 
          for i in range(len(categorias))]

bars = ax.barh(categorias, precios, color=colors, height=0.6, edgecolor="none")

# Agregar valor al final de cada barra
for i, (bar, precio, total) in enumerate(zip(bars, precios, total_libros)):
    ax.text(
        bar.get_width() + 0.1,
        bar.get_y() + bar.get_height() / 2,
        f"£{precio:.2f} ({total} libros)",
        va="center", ha="left", fontsize=9, color="#444441"
    )

# Estilo
ax.set_xlabel("Precio promedio (£)", fontsize=11)
ax.set_title("Top 10 categorías por precio promedio", fontsize=14, fontweight="bold", pad=15)
ax.invert_yaxis()  # la categoría más cara arriba
ax.set_xlim(0, max(precios) * 1.3)  # espacio para las etiquetas
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.tick_params(axis="y", labelsize=10)

plt.tight_layout()
plt.savefig("outputs/top_categorias_precio.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Gráfico guardado en outputs/top_categorias_precio.png")


## 9. Diagrama UML del Modelo de Datos

El diagrama UML (Unified Modeling Language) muestra visualmente la estructura
de la base de datos y las relaciones entre tablas.

**Notación:**
- `PK` → Primary Key (clave primaria, identificador único)
- `FK` → Foreign Key (clave foránea, referencia a otra tabla)
- `1` → lado "uno" de la relación
- `*` → lado "muchos" de la relación

**Relaciones:**
- `categories` 1 ──── * `books` → una categoría tiene muchos libros
- `books` * ──── * `authors` → relación muchos a muchos (resuelta con `book_author`)


In [ ]:
# Diagrama UML textual del modelo de datos
uml = """
┌─────────────────────┐         ┌──────────────────────────┐
│      categories     │         │          books           │
├─────────────────────┤         ├──────────────────────────┤
│ PK  id    INTEGER   │────┐    │ PK  id          INTEGER  │
│     name  TEXT      │    │    │     title       TEXT     │
│     slug  TEXT      │    └───>│ FK  category_id INTEGER  │
│     created_at      │    1  * │     price       REAL     │
└─────────────────────┘         │     rating      INTEGER  │
                                │     url         TEXT     │
                                │     description TEXT     │
                                │     created_at           │
                                └──────────┬───────────────┘
                                           │ *
                                           │
                                ┌──────────┴───────────────┐
                                │       book_author        │
                                │     (tabla intermedia)   │
                                ├──────────────────────────┤
                                │ FK  book_id    INTEGER   │
                                │ FK  author_id  INTEGER   │
                                │ PK  (book_id, author_id) │
                                └──────────┬───────────────┘
                                           │ *
                                           │
                                ┌──────────┴───────────────┐
                                │         authors          │
                                ├──────────────────────────┤
                                │ PK  id               INT │
                                │     name            TEXT │
                                │     birth_year       INT │
                                │     country         TEXT │
                                │     external_api_id TEXT │
                                │     total_known_works INT│
                                │     api_source      TEXT │
                                │     api_status      TEXT │
                                │     created_at           │
                                └──────────────────────────┘
"""
print(uml)


## 10. Cierre del Pipeline

### Resumen de lo que construimos:

1. **Scraping** → extrajimos todas las categorías y libros de Books to Scrape usando `requests` + `BeautifulSoup`
2. **API** → enriquecimos los autores con Open Library, implementando cache de dos niveles
3. **Base de datos** → modelo relacional normalizado con 4 tablas y relación M:N
4. **SQL** → 5 consultas desde filtros simples hasta JOINs de 3 tablas con GROUP BY
5. **Performance** → demostración de impacto de indexación con medición antes/después
6. **Visualización** → gráfico de barras con matplotlib

### Buenas prácticas aplicadas:
- Control de errores en cada capa (scraping, API, DB)
- Cache para no repetir llamadas a la API
- Parámetros en SQL para prevenir SQL injection
- `INSERT OR IGNORE` para idempotencia (se puede correr múltiples veces sin duplicar)
- Delays entre requests (scraping ético)


In [ ]:
# Cerrar la conexión a la base de datos
conn.close()
print("✅ Pipeline completado exitosamente")
print("   Conexión a la base de datos cerrada")
print()
print("📁 Archivos generados:")
import os
for f in ["books.db", "authors_cache.json", "outputs/top_categorias_precio.png"]:
    exists = "✅" if os.path.exists(f) else "❌"
    print(f"   {exists} {f}")
